# Phase 4 — Session 3 v5 Part 1: TrOCR Full Fine-Tune + Hard-Word Training

## What's new in v5 (over v4)
1. **No label smoothing** — v4's `label_smoothing=0.1` created a loss floor
   at 0.813; the model was already at 0.87 and couldn't improve further.
2. **Full fine-tuning** instead of LoRA — differential LR for encoder/decoder,
   mirroring Phase 3.5's proven strategy.
3. **Hard-word training** — NW gap penalty increased (-1 → -4) + mismatch
   penalty capped at 3 so hard words get MATCHED instead of skipped.
   All aligned pairs (edit_ratio ≤ 0.60) included with quality-weighted loss.
4. **2-Phase training** — Phase A: encoder frozen, exact-match pairs (warm-up);
   Phase B: full fine-tune, ALL pairs with quality weights.
5. **Lighter augmentation** — removed aggressive distortions on already-noisy crops.
6. **Checkpoint by page CER** — more reliable than val loss.

**Target: CER < 0.10, WER < 0.15 (Char Acc 90%+, Word Acc 85%+)**

## Install Dependencies

In [3]:
!pip install -q "python-doctr[torch]>=1.0" "transformers>=4.40,<5.0" \
    "peft>=0.10" "accelerate>=0.30" "albumentations==1.3.1" \
    "editdistance>=0.6" "pillow>=9.0" "opencv-python-headless>=4.8" \
    "matplotlib>=3.7" "numpy>=1.24" "pandas>=1.5" "scipy>=1.10"

import importlib
for _p in ["doctr", "cv2", "PIL", "numpy", "scipy"]:
    _m = importlib.import_module(_p)
    print(f"  {_p}: {getattr(_m, '__version__', 'ok')}")
print("All packages ready.")

  doctr: v1.0.1
  cv2: 4.13.0
  PIL: 11.3.0
  numpy: 2.4.6
  scipy: 1.16.3
All packages ready.


## Imports, Paths, Config

In [4]:
import os, json, time, random, math, unicodedata, warnings, shutil, gc, glob
from pathlib import Path
from io import BytesIO
from collections import Counter, defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
import scipy.ndimage as ndi
import torch, torch.nn as nn, torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import editdistance, albumentations as A
from transformers import (
    VisionEncoderDecoderModel, TrOCRProcessor, get_scheduler
)
os.environ["USE_TORCH"] = "1"
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path("/kaggle/working/phase4")
CKPT_DIR        = OUTPUT_DIR / "checkpoints"
CROPS_DIR       = OUTPUT_DIR / "word_crops"
PSEUDO_DIR      = OUTPUT_DIR / "pseudo_labels"
RESULTS_DIR     = OUTPUT_DIR / "results"

BEST_DET_CKPT   = Path("/kaggle/input/models/nehamalik10/phase-4-dbnet-manual/pytorch/default/1/dbnet_manual_best.pt")
SESSION2_ARTIFACTS = Path("/kaggle/input/datasets/nehamalik10/phase4-session2-outputs/session2_artifacts.json")
PHASE2_DIR      = Path("/kaggle/input/datasets/nehamalik10/hindi-htr-phase2-artifacts-improved")
DATA_DIR        = Path("/kaggle/input/datasets/nehamalik10/hindi-page-level-dataset")
TROCR_CKPT      = Path("/kaggle/input/models/nehamalik10/phase-3-5-model/pytorch/default/1/best_model_trocr_final.pt")

for d in [OUTPUT_DIR, CKPT_DIR, CROPS_DIR, PSEUDO_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Load Session 2 Artifacts ──────────────────────────────────────────────────
_art_path = SESSION2_ARTIFACTS
if not _art_path.exists():
    _art_path = OUTPUT_DIR / "session2_artifacts.json"
if not _art_path.exists():
    raise FileNotFoundError("Session 2 artifacts not found")
with open(_art_path, "r", encoding="utf-8") as f:
    art = json.load(f)
print(f"Loaded artifacts from: {_art_path}")

if not BEST_DET_CKPT.exists():
    raise FileNotFoundError("DBNet checkpoint not found")

# ── Vocab ─────────────────────────────────────────────────────────────────────
char_to_token = art["char_to_token"]
token_to_char = {int(k): v for k, v in art["token_to_char"].items()}
actual_chars  = art["actual_chars"]
VOCAB_SIZE    = art["VOCAB_SIZE"]   # 140
PAD_ID  = art["PAD_ID"]   # 0
BOS_ID  = art["BOS_ID"]   # 1
EOS_ID  = art["EOS_ID"]   # 2
UNK_ID  = art["UNK_ID"]   # 3
print(f"PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}, UNK={UNK_ID}")

# ── Detection Config ──────────────────────────────────────────────────────────
DET_IMG_SIZE     = 1024
BIN_THRESH       = 0.50
BOX_THRESH       = 0.06
MIN_WORD_H       = 8
MIN_WORD_W       = 6
CROP_PAD         = 4

# ── TrOCR Config ──────────────────────────────────────────────────────────────
TROCR_SIZE       = 384
MAX_LEN          = 48
NO_REPEAT_NGRAM  = 0

# ── Training Config ── ★ KEY v5 CHANGES ──────────────────────────────────────
LABEL_SMOOTH  = 0.0     # ★ v5: was 0.1 — removed loss floor
GRAD_ACCUM    = 4       # ★ v5: was 8 — more frequent updates
FT_BS         = 8
FT_WD         = 1e-4
FT_GRAD_CLIP  = 1.0
FT_PATIENCE   = 5       # ★ v5: was 3

# Phase A: encoder frozen, exact-match pairs only
ENCODER_LR_A  = 0.0     # frozen
DECODER_LR_A  = 5e-5
PHASE_A_EPOCHS = 5
PHASE_A_MAX_EDIT = 0.05  # near-exact matches only

# Phase B: full fine-tune, all pairs with quality weights
ENCODER_LR_B  = 2e-5
DECODER_LR_B  = 5e-5
PHASE_B_EPOCHS = 12
PHASE_B_MAX_EDIT = 1.0   # all pairs (quality weights handle noise)

# ── NW Alignment Config ── ★ KEY v5 CHANGES ──────────────────────────────────
NW_GAP           = -4     # ★ v5: was -1 — forces matching over gaps
NW_MATCH         = 2
NW_MAX_MISMATCH  = 3      # ★ v5: NEW — caps mismatch penalty
POSITION_THRESH  = 0.35
PSEUDO_MAX_EDIT  = 0.60   # ★ v5: was 0.30 — include hard words

det_metrics = art.get("det_metrics", {})
print(f"Config loaded. Det F1: {det_metrics.get('f1','N/A')}")

2026-06-17 07:23:58.983192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781681039.214056      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781681039.274776      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781681039.815692      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781681039.815736      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781681039.815739      58 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Loaded artifacts from: /kaggle/input/datasets/nehamalik10/phase4-session2-outputs/session2_artifacts.json
PAD=0, BOS=1, EOS=2, UNK=3
Config loaded. Det F1: 0.9405099150141644


## Utility Functions + Phase 3.5 Tight-Crop + Load Data

In [5]:
def encode(text):
    t = unicodedata.normalize("NFC", text)
    return [char_to_token.get(c, UNK_ID) for c in t] + [EOS_ID]

def decode_ids(ids):
    out = []
    for tok in ids:
        tok = int(tok)
        if tok == EOS_ID: break
        if tok in (PAD_ID, BOS_ID): continue
        out.append(token_to_char.get(tok, ""))
    return unicodedata.normalize("NFC", "".join(out))

def _detect_cols(df):
    cols = [c.lower() for c in df.columns]
    ic = next((df.columns[i] for i, c in enumerate(cols)
               if any(k in c for k in ("image", "img", "pixel"))), df.columns[0])
    tc = next((df.columns[i] for i, c in enumerate(cols)
               if any(k in c for k in ("text", "transcript", "label"))), df.columns[-1])
    return ic, tc

def pil_page(page):
    raw = page.get("image_bytes") or page.get("image")
    if raw is None: return Image.new("RGB", (512, 512), 255)
    try: return Image.open(BytesIO(raw)).convert("RGB")
    except: return Image.new("RGB", (512, 512), 255)

def load_pages(parquet_path):
    df = pd.read_parquet(parquet_path)
    ic, tc = _detect_cols(df)
    pages = []
    for _, row in df.iterrows():
        img_data = row[ic]
        img_bytes = img_data.get("bytes", None) if isinstance(img_data, dict) else img_data
        text = str(row[tc]) if pd.notna(row[tc]) else ""
        pages.append({"image_bytes": img_bytes, "text": text.strip()})
    return pages

def autocontrast(gray, clip=1.0):
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    c = clip / 100 * float(gray.size)
    cs = np.cumsum(hist)
    lo = int(np.searchsorted(cs, c))
    hi = int(255 - np.searchsorted(cs[::-1], c))
    if hi <= lo: return gray
    return np.clip((gray.astype(np.float32) - lo) * 255 / (hi - lo), 0, 255).astype(np.uint8)

def deskew(gray, max_deg=5.0):
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, 100)
    if lines is None: return gray
    angle = float(np.median([l[0][1] * 180 / np.pi - 90 for l in lines]))
    if abs(angle) > max_deg: return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), borderValue=255)

def preprocess_page(img):
    arr = np.array(img)
    if arr.ndim == 3 and arr.shape[2] == 4: arr = arr[:, :, :3]
    gray = arr if arr.ndim == 2 else cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    gray = autocontrast(gray)
    gray = deskew(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

# ── Phase 3.5 Tight-Crop Preprocessing ───────────────────────────────────────
_trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")

def preprocess_crop_for_trocr(crop_pil):
    gray = np.array(crop_pil.convert("L"))
    p2, p98 = np.percentile(gray, (2, 98))
    if p98 > p2:
        gray = np.clip((gray - p2) / (p98 - p2) * 255, 0, 255).astype(np.uint8)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(bw)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        y1, y2 = max(0, y - 2), min(gray.shape[0], y + h + 2)
        x1, x2 = max(0, x - 2), min(gray.shape[1], x + w + 2)
        gray = gray[y1:y2, x1:x2]
    if gray.shape[0] < 4 or gray.shape[1] < 4:
        gray = np.full((32, 128), 255, dtype=np.uint8)
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    pil_rgb = Image.fromarray(rgb)
    w, h = pil_rgb.size
    scale = min(TROCR_SIZE / w, TROCR_SIZE / h)
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resized = pil_rgb.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new("RGB", (TROCR_SIZE, TROCR_SIZE), (255, 255, 255))
    canvas.paste(resized, ((TROCR_SIZE - nw) // 2, (TROCR_SIZE - nh) // 2))
    return _trocr_processor(images=canvas, return_tensors="pt").pixel_values.squeeze(0)

# ── Load data ─────────────────────────────────────────────────────────────────
train_pages = load_pages(DATA_DIR / "line_train.parquet")
val_pages   = load_pages(DATA_DIR / "line_val.parquet")
test_pages  = load_pages(DATA_DIR / "line_test.parquet")
print(f"Data: train={len(train_pages):,}  val={len(val_pages):,}  test={len(test_pages):,}")
print("Utilities ready.")

preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Data: train=3,040  val=380  test=380
Utilities ready.


## Load DBNet + Detection

In [6]:
from doctr.models import db_resnet50
from doctr.models.detection.predictor import DetectionPredictor
from doctr.models.preprocessor import PreProcessor

def _parse_det_output(raw):
    if isinstance(raw, dict):
        raw = raw.get("words", raw.get("boxes", np.zeros((0, 5), dtype=np.float32)))
    arr = np.asarray(raw)
    if arr.ndim == 2 and arr.shape[1] >= 5:
        return arr.astype(np.float32)
    return np.zeros((0, 5), dtype=np.float32)

def _load_dbnet(ckpt_path):
    det = db_resnet50(pretrained=False)
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if isinstance(state, dict) and "model" in state:
        state = state["model"]
    det.load_state_dict(state)
    det.postprocessor.bin_thresh = BIN_THRESH
    det.postprocessor.box_thresh = BOX_THRESH
    det = det.to(DEVICE).eval()
    pre = PreProcessor(
        (DET_IMG_SIZE, DET_IMG_SIZE), batch_size=1,
        mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0),
        preserve_aspect_ratio=False,
    )
    return det, DetectionPredictor(pre_processor=pre, model=det)

det_model, DET_PREDICTOR = _load_dbnet(BEST_DET_CKPT)
print(f"DBNet loaded from: {BEST_DET_CKPT}")

def detect_words(predictor, img):
    arr = np.array(img)
    if arr.ndim == 2: arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    elif arr.ndim == 3 and arr.shape[2] == 4: arr = arr[:, :, :3]
    if arr.dtype != np.uint8: arr = np.clip(arr, 0, 255).astype(np.uint8)
    h, w = arr.shape[:2]
    page_area = h * w
    raw = _parse_det_output(predictor([arr])[0])
    boxes = []
    for row in raw:
        x1, y1 = int(float(row[0]) * w), int(float(row[1]) * h)
        x2, y2 = int(float(row[2]) * w), int(float(row[3]) * h)
        conf = float(row[4])
        bw, bh = x2 - x1, y2 - y1
        if bh < MIN_WORD_H or bw < MIN_WORD_W: continue
        if bw * bh > page_area * 0.25: continue
        boxes.append({"x1": x1, "y1": y1, "x2": x2, "y2": y2,
                       "confidence": round(conf, 4)})
    boxes.sort(key=lambda b: (b["y1"], b["x1"]))
    if len(boxes) < 3:
        proj = _detect_projection(arr)
        if len(proj) > len(boxes): boxes = proj
    return boxes

def _detect_projection(arr):
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY) if arr.ndim == 3 else arr
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    hproj = binary.sum(axis=1).astype(np.float32)
    hs = ndi.uniform_filter1d(hproj, 5)
    thresh = hs.max() * 0.05
    lines, in_line, y_start = [], False, 0
    for y, v in enumerate(hs):
        if v > thresh and not in_line: in_line = True; y_start = y
        elif v <= thresh and in_line:
            in_line = False
            if y - y_start > 8: lines.append((y_start, y))
    if in_line and len(hs) - y_start > 8: lines.append((y_start, len(hs)))
    boxes = []
    for y1, y2 in lines:
        vproj = binary[y1:y2, :].sum(axis=0).astype(np.float32)
        vs = ndi.uniform_filter1d(vproj, 3)
        vt = vs.max() * 0.05
        in_w, xs = False, 0
        for x, v in enumerate(vs):
            if v > vt and not in_w: in_w = True; xs = x
            elif v <= vt and in_w:
                in_w = False
                if x - xs > 6:
                    boxes.append({"x1": xs, "y1": y1, "x2": x, "y2": y2, "confidence": 0.5})
        if in_w and len(vs) - xs > 6:
            boxes.append({"x1": xs, "y1": y1, "x2": len(vs), "y2": y2, "confidence": 0.5})
    return sorted(boxes, key=lambda b: (b["y1"], b["x1"]))

print("Detection ready (NO IoU merge).")

DBNet loaded from: /kaggle/input/models/nehamalik10/phase-4-dbnet-manual/pytorch/default/1/dbnet_manual_best.pt
Detection ready (NO IoU merge).


## Load TrOCR (★ FULL MODEL, NO LoRA)

In [7]:
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten")

# Fix config at ALL levels
model.config.pad_token_id = PAD_ID
model.config.decoder_start_token_id = BOS_ID
model.config.eos_token_id = EOS_ID
model.config.bos_token_id = BOS_ID
model.decoder.config.pad_token_id = PAD_ID
model.decoder.config.bos_token_id = BOS_ID
model.decoder.config.eos_token_id = EOS_ID
model.decoder.config.decoder_start_token_id = BOS_ID

# Resize decoder embeddings for Devanagari vocab
model.decoder.resize_token_embeddings(VOCAB_SIZE)
print(f"Decoder vocab resized to {VOCAB_SIZE}")

# Load Phase 3.5 weights
if not TROCR_CKPT.exists():
    raise FileNotFoundError(f"Phase 3.5 checkpoint not found: {TROCR_CKPT}")
_state = torch.load(TROCR_CKPT, map_location="cpu", weights_only=False)
if isinstance(_state, dict) and "model_state_dict" in _state:
    _state = _state["model_state_dict"]
_md = model.state_dict()
_loaded, _skip = 0, 0
for k, v in _state.items():
    if k in _md and v.shape == _md[k].shape:
        _md[k] = v; _loaded += 1
    else:
        _skip += 1
model.load_state_dict(_md)
print(f"★ Phase 3.5 weights: {_loaded}/{len(_state)} loaded, {_skip} skipped")

# Sync tokenizer
_tok = PHASE2_DIR / "char_vocab.json"
if _tok.exists():
    with open(_tok, encoding="utf-8") as f:
        _raw = json.load(f)
    if isinstance(_raw, dict) and "char_to_token" in _raw:
        for k, v in _raw["char_to_token"].items(): char_to_token[k] = v
        token_to_char.update({v: k for k, v in char_to_token.items()})
        print("  Tokenizer synced.")

# ★ v5: NO LoRA — full fine-tuning
_total = sum(p.numel() for p in model.parameters())
_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n★ FULL FINE-TUNE: {_train:,} / {_total:,} params ({_train/_total*100:.1f}%)")

# Fix generation config
if hasattr(model, 'generation_config'):
    model.generation_config.max_length = MAX_LEN
    model.generation_config.num_beams = 1
    model.generation_config.no_repeat_ngram_size = NO_REPEAT_NGRAM
    model.generation_config.decoder_start_token_id = BOS_ID
    model.generation_config.pad_token_id = PAD_ID
    model.generation_config.eos_token_id = EOS_ID

model = model.to(DEVICE).eval()

print(f"  pad_token_id={model.config.pad_token_id} (expect {PAD_ID})")
print(f"  decoder_start_token_id={model.config.decoder_start_token_id} (expect {BOS_ID})")
print(f"  eos_token_id={model.config.eos_token_id} (expect {EOS_ID})")
assert model.config.pad_token_id == PAD_ID
assert model.config.decoder_start_token_id == BOS_ID
assert model.config.eos_token_id == EOS_ID
print(f"TrOCR (Phase 3.5, full fine-tune) ready on {DEVICE}.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/246M [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Decoder vocab resized to 140


model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

★ Phase 3.5 weights: 362/362 loaded, 0 skipped

★ FULL FINE-TUNE: 28,877,824 / 28,877,824 params (100.0%)
  pad_token_id=0 (expect 0)
  decoder_start_token_id=1 (expect 1)
  eos_token_id=2 (expect 2)
TrOCR (Phase 3.5, full fine-tune) ready on cuda.


## Recognition Pipeline + Line Assembly

In [8]:
_hindi_vocab = Counter()
_vocab_ready = False

def build_vocab():
    global _hindi_vocab, _vocab_ready
    if _vocab_ready: return
    _hindi_vocab = Counter(
        w for p in train_pages for w in p["text"].split() if len(w) >= 2
    )
    _vocab_ready = True
    print(f"  Vocab: {len(_hindi_vocab):,} unique words")

def _correct_word(w):
    if not _hindi_vocab or len(w) > 12 or len(w) < 2: return w
    if w in _hindi_vocab and _hindi_vocab[w] >= 3: return w
    bw, be, bf = w, 99, 0
    wl = list(w)
    for cand, freq in _hindi_vocab.most_common(20000):
        if abs(len(cand) - len(w)) > 1: continue
        ed = editdistance.eval(wl, list(cand))
        if ed == 0: return cand
        if ed <= 1 and (ed < be or (ed == be and freq > bf)):
            be, bw, bf = ed, cand, freq
    return bw if be <= 1 and bf >= 3 else w

_DEP = "\u093e\u093f\u0940\u0941\u0942\u0943\u0944\u0947\u0948\u094b\u094c"
def fix_deva(t):
    for c in _DEP + "\u094d":
        t = t.replace(f" {c}", c)
    return t

def assemble(word_texts, boxes, correct=False):
    if not boxes or not word_texts: return ""
    items = list(zip(word_texts, boxes))
    items.sort(key=lambda tb: ((tb[1]["y1"] + tb[1]["y2"]) / 2.0, tb[1]["x1"]))
    heights = [b["y2"] - b["y1"] for _, b in items]
    med_h = max(10, float(np.median(heights)))
    line_thresh = med_h * 0.7
    lines = [[items[0]]]
    line_yc = (items[0][1]["y1"] + items[0][1]["y2"]) / 2.0
    for t, b in items[1:]:
        yc = (b["y1"] + b["y2"]) / 2.0
        if abs(yc - line_yc) > line_thresh:
            lines.append([(t, b)]); line_yc = yc
        else:
            lines[-1].append((t, b))
            line_yc = np.mean([(bb["y1"] + bb["y2"]) / 2.0 for _, bb in lines[-1]])
    result = []
    for line in lines:
        line.sort(key=lambda tb: tb[1]["x1"])
        result.append(" ".join(t for t, _ in line))
    text = fix_deva(unicodedata.normalize("NFC", "\n".join(result)))
    if correct:
        build_vocab()
        text = " ".join(_correct_word(w) for w in text.split())
    return text

@torch.inference_mode()
def recognize_batch(pv_list, batch_size=32, num_beams=1):
    if not pv_list: return []
    out = []
    for i in range(0, len(pv_list), batch_size):
        batch = torch.stack(pv_list[i:i + batch_size]).to(DEVICE)
        ids = model.generate(
            pixel_values=batch, max_length=MAX_LEN, num_beams=num_beams,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            decoder_start_token_id=BOS_ID,
            pad_token_id=PAD_ID, eos_token_id=EOS_ID
        )
        out.extend(decode_ids(seq) for seq in ids)
    return out

def crop_words(img, boxes, pad=CROP_PAD):
    arr = np.array(img); H, W = arr.shape[:2]
    crops = []
    for b in boxes:
        x1, y1 = max(0, b["x1"] - pad), max(0, b["y1"] - pad)
        x2, y2 = min(W, b["x2"] + pad), min(H, b["y2"] + pad)
        c = arr[y1:y2, x1:x2]
        crops.append(Image.fromarray(c) if c.size > 0 else Image.new("RGB", (32, 32), 255))
    return crops

def recognize_page(page, use_beam=False, use_correction=False):
    img = pil_page(page)
    arr = preprocess_page(img)
    img = Image.fromarray(arr)
    boxes = detect_words(DET_PREDICTOR, img)
    if not boxes:
        return {"text": "", "word_texts": [], "word_boxes": [], "n_words": 0}
    crops = crop_words(img, boxes)
    pvs = [preprocess_crop_for_trocr(c) for c in crops]
    wt = recognize_batch(pvs, num_beams=4 if use_beam else 1)
    pt = assemble(wt, boxes, correct=use_correction)
    return {"text": pt, "word_texts": wt, "word_boxes": boxes, "n_words": len(boxes)}

def compute_metrics(pred, gt):
    p = unicodedata.normalize("NFC", pred.strip())
    g = unicodedata.normalize("NFC", gt.strip())
    if not g: return {"cer": 1.0 if p else 0.0, "wer": 1.0 if p else 0.0}
    cer = editdistance.eval(list(p), list(g)) / max(len(g), 1)
    wer = editdistance.eval(p.split(), g.split()) / max(len(g.split()), 1)
    return {"cer": cer, "wer": wer}

@torch.inference_mode()
def quick_page_eval(n=20):
    """Evaluate page-level CER/WER on first n val pages."""
    model.eval()
    cers, wers = [], []
    for i in range(min(n, len(val_pages))):
        gt = (val_pages[i].get("text") or "").strip()
        if not gt: continue
        try:
            res = recognize_page(val_pages[i])
        except:
            continue
        m = compute_metrics(res["text"], gt)
        cers.append(m["cer"]); wers.append(m["wer"])
    if not cers: return 1.0, 1.0
    return float(np.mean(cers)), float(np.mean(wers))

print("Recognition pipeline ready.")

Recognition pipeline ready.


## Baseline Evaluation

In [9]:
print("=" * 70)
print("BASELINE (Phase 3.5, before fine-tuning)")
print("=" * 70)
model.eval()

print("\n── Diagnostic (first 3 pages) ──")
for _i in range(min(3, len(val_pages))):
    _p = val_pages[_i]
    _gt = (_p.get("text") or "").strip()
    if not _gt: continue
    try: _r = recognize_page(_p)
    except Exception as e: print(f"  Page {_i} error: {e}"); continue
    _m = compute_metrics(_r["text"], _gt)
    print(f"\n  Page {_i}: {_r['n_words']} words, CER={_m['cer']:.4f}, WER={_m['wer']:.4f}")
    print(f"    GT  : {' '.join(_gt.split()[:6])}")
    print(f"    Pred: {' '.join(_r['text'].split()[:6])}")

print("\n── Baseline on 50 val pages ──")
_cers, _wers = [], []
for _i in range(min(50, len(val_pages))):
    _gt = (val_pages[_i].get("text") or "").strip()
    if not _gt: continue
    try: _r = recognize_page(val_pages[_i])
    except: continue
    _m = compute_metrics(_r["text"], _gt)
    _cers.append(_m["cer"]); _wers.append(_m["wer"])
    if (_i + 1) % 10 == 0:
        print(f"  [{_i+1}/50] CER={np.mean(_cers):.4f}  WER={np.mean(_wers):.4f}")

baseline_cer = float(np.mean(_cers))
baseline_wer = float(np.mean(_wers))
print(f"\nBASELINE: CER={baseline_cer:.4f}  WER={baseline_wer:.4f}")
print(f"  Char Acc: {(1-baseline_cer)*100:.2f}%  Word Acc: {(1-baseline_wer)*100:.2f}%")

BASELINE (Phase 3.5, before fine-tuning)

── Diagnostic (first 3 pages) ──

  Page 0: 40 words, CER=0.2667, WER=0.4634
    GT  : कोशिका आम तौर पर सक्रिय के
    Pred: कोशिका आम तौर पर सक्रिय के

  Page 1: 72 words, CER=0.1621, WER=0.2800
    GT  : १३ साल की उम्र वह वेस्टर्न
    Pred: २३ साल की उम्र वह देखने

  Page 2: 34 words, CER=0.6078, WER=0.8039
    GT  : अब्द-अर-रहमान तृतीय ने दुनिया को प्रभावित
    Pred: अद्द रहमात हवीय ते दुनिया को

── Baseline on 50 val pages ──
  [10/50] CER=0.3136  WER=0.5000
  [20/50] CER=0.2999  WER=0.4810
  [30/50] CER=0.3591  WER=0.5693
  [40/50] CER=0.3179  WER=0.5172
  [50/50] CER=0.3166  WER=0.5159

BASELINE: CER=0.3166  WER=0.5159
  Char Acc: 68.34%  Word Acc: 48.41%


## Pseudo-Label Generation (★ v5: Relaxed Filtering + Hard Words)

Key changes from v4:
- **NW gap penalty: -4** (was -1) — forces matching hard words instead of skipping
- **Mismatch cap: 3** — prevents gap preference for high edit-distance pairs
- **max_edit = 0.60** (was 0.30) — includes hard words in training
- **Quality weight** saved for each pair: `confidence × (1 - edit_ratio)`

**~3 hours on T4 for 3040 pages**

## Pseudo-Label Generation

In [10]:
def needleman_wunsch(pw, gw):
    m, n = len(pw), len(gw)
    dp = np.full((m + 1, n + 1), -1e9, dtype=np.float64)
    dp[0, 0] = 0.0
    for i in range(1, m + 1): dp[i, 0] = dp[i - 1, 0] + NW_GAP
    for j in range(1, n + 1): dp[0, j] = dp[0, j - 1] + NW_GAP

    def sim(a, b):
        if a == b: return float(NW_MATCH)
        ed = editdistance.eval(list(a), list(b))
        return -min(float(ed), NW_MAX_MISMATCH)  # ★ v5: cap mismatch

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i, j] = max(
                dp[i - 1, j - 1] + sim(pw[i - 1], gw[j - 1]),
                dp[i - 1, j] + NW_GAP,
                dp[i, j - 1] + NW_GAP
            )

    align, i, j = [], m, n
    while i > 0 or j > 0:
        if i == 0:
            align.append((None, j - 1)); j -= 1
        elif j == 0:
            align.append((i - 1, None)); i -= 1
        else:
            s = sim(pw[i - 1], gw[j - 1])
            if abs(dp[i, j] - (dp[i - 1, j - 1] + s)) < 1e-9:
                align.append((i - 1, j - 1)); i -= 1; j -= 1
            elif abs(dp[i, j] - (dp[i - 1, j] + NW_GAP)) < 1e-9:
                align.append((i - 1, None)); i -= 1
            else:
                align.append((None, j - 1)); j -= 1
    return list(reversed(align))

def validate_pair(pi, gi, n_pred, n_gt, pred_word, gt_word):
    pred_ratio = pi / max(n_pred - 1, 1)
    gt_ratio = gi / max(n_gt - 1, 1)
    if abs(pred_ratio - gt_ratio) > POSITION_THRESH: return False
    if len(gt_word) >= 2 and len(pred_word) >= 1:
        ratio = len(pred_word) / len(gt_word)
        if ratio > 3.0 or ratio < 0.33: return False
    if len(gt_word) < 1 or len(pred_word) < 1: return False
    return True

def generate_pseudo_labels(pages, max_edit=PSEUDO_MAX_EDIT, label="R1"):
    crop_dir = CROPS_DIR / f"pseudo_{label}"
    crop_dir.mkdir(exist_ok=True)
    resume_path = PSEUDO_DIR / f"pseudo_pairs_{label}.json"

    pairs, done = [], set()
    if resume_path.exists():
        with open(resume_path, encoding="utf-8") as f:
            pairs = json.load(f)
        done = {p["page_idx"] for p in pairs}
        print(f"  Resumed: {len(pairs):,} pairs from {len(done):,} pages")

    stats = {"total": 0, "pos_rej": 0, "edit_rej": 0, "accepted": 0}
    t0 = time.time()

    for idx, page in enumerate(pages):
        if idx in done: continue
        gt = (page.get("text") or "").strip()
        if not gt: continue
        try: res = recognize_page(page)
        except: continue
        pw, bx, gw = res["word_texts"], res["word_boxes"], gt.split()
        if not pw or not gw: continue

        align = needleman_wunsch(pw, gw)
        img = pil_page(page)
        n_pred, n_gt = len(pw), len(gw)

        for pi, gi in align:
            if pi is None or gi is None: continue
            if gi >= len(gw) or pi >= len(bx): continue
            g, p_ = gw[gi], pw[pi]
            stats["total"] += 1

            if not validate_pair(pi, gi, n_pred, n_gt, p_, g):
                stats["pos_rej"] += 1; continue

            ed = editdistance.eval(list(p_), list(g))
            ratio = ed / max(len(g), 1)
            if ratio > max_edit:
                stats["edit_rej"] += 1; continue

            b = bx[pi]; arr = np.array(img); H, W = arr.shape[:2]
            x1, y1 = max(0, b["x1"] - CROP_PAD), max(0, b["y1"] - CROP_PAD)
            x2, y2 = min(W, b["x2"] + CROP_PAD), min(H, b["y2"] + CROP_PAD)
            crop = arr[y1:y2, x1:x2]
            if crop.size == 0: continue

            cn = f"p{idx:05d}_w{pi:03d}.png"
            Image.fromarray(crop).save(crop_dir / cn)
            q = round(b["confidence"] * (1 - ratio), 4)
            pairs.append({
                "page_idx": idx, "word_idx": pi, "crop_file": cn,
                "gt_text": g, "pred_text": p_,
                "edit_ratio": round(ratio, 4),
                "confidence": round(b["confidence"], 4),
                "quality": max(0.1, q),  # ★ v5: min quality 0.1
                "pred_pos": pi, "gt_pos": gi,
            })
            stats["accepted"] += 1

        done.add(idx)
        if (idx + 1) % 200 == 0:
            with open(resume_path, "w", encoding="utf-8") as f:
                json.dump(pairs, f, ensure_ascii=False)
            el = time.time() - t0
            print(f"  [{idx+1}/{len(pages)}] {len(pairs):,} pairs ({el/60:.1f}min)")

    with open(resume_path, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False)

    el = time.time() - t0
    print(f"\n  {label}: {len(pairs):,} pairs from {len(done):,} pages ({el/60:.1f}min)")
    print(f"  Total: {stats['total']:,} | PosRej: {stats['pos_rej']:,} "
          f"| EditRej: {stats['edit_rej']:,} | Accepted: {stats['accepted']:,}")
    if pairs:
        er = [p["edit_ratio"] for p in pairs]
        print(f"  Avg edit ratio: {np.mean(er):.3f}")
        for t in [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
            n = sum(1 for e in er if e <= t)
            print(f"    ≤{t:.2f}: {n:,} ({n/len(er)*100:.1f}%)")
    return pairs, crop_dir

print("Generating pseudo-labels (★ v5: relaxed filtering, hard words included)...")
model.eval()
pseudo_pairs, CROP_DIR_R1 = generate_pseudo_labels(train_pages, max_edit=PSEUDO_MAX_EDIT, label="R1")

Generating pseudo-labels (★ v5: relaxed filtering, hard words included)...
  [200/3040] 4,086 pairs (10.0min)
  [400/3040] 8,407 pairs (20.4min)
  [600/3040] 12,648 pairs (30.9min)
  [800/3040] 17,079 pairs (41.9min)
  [1000/3040] 20,992 pairs (52.2min)
  [1200/3040] 25,358 pairs (62.8min)
  [1400/3040] 29,749 pairs (73.7min)
  [1600/3040] 33,799 pairs (83.5min)
  [1800/3040] 37,789 pairs (93.8min)
  [2000/3040] 41,673 pairs (104.1min)
  [2200/3040] 45,908 pairs (114.7min)
  [2400/3040] 49,830 pairs (124.9min)
  [2600/3040] 54,114 pairs (135.3min)
  [2800/3040] 58,059 pairs (145.6min)
  [3000/3040] 62,380 pairs (156.3min)

  R1: 63,199 pairs from 3,040 pages (158.4min)
  Total: 135,389 | PosRej: 6,627 | EditRej: 65,563 | Accepted: 63,199
  Avg edit ratio: 0.091
    ≤0.00: 47,770 (75.6%)
    ≤0.05: 47,770 (75.6%)
    ≤0.10: 47,841 (75.7%)
    ≤0.20: 50,804 (80.4%)
    ≤0.30: 53,237 (84.2%)
    ≤0.40: 56,044 (88.7%)
    ≤0.50: 62,313 (98.6%)
    ≤0.60: 63,199 (100.0%)


## Training (★ v5: Full Fine-Tune, Quality-Weighted, No Label Smoothing)

**Phase A** (encoder frozen, ~1 hr): Warm up decoder on exact-match pairs only.
**Phase B** (full fine-tune, ~5 hrs): Train ALL parameters on ALL pairs with
quality-weighted loss. Hard words (high edit_ratio) get lower weight.

Checkpointing by **page-level CER** (not loss).

## Dataset + Training Helpers

In [11]:
class PseudoDS(Dataset):
    def __init__(self, pairs, crop_dir, augment=False):
        self.pairs = pairs
        self.crop_dir = Path(crop_dir)
        self.augment = augment
        if augment:
            # ★ v5: lighter augmentation — no elastic/grid/optical distortion
            self.aug = A.Compose([
                A.OneOf([
                    A.GaussNoise(var_limit=(5, 25), p=0.5),
                    A.MotionBlur(blur_limit=3, p=0.5),
                ], p=0.3),
                A.CLAHE(clip_limit=1.5, p=0.2),
                A.RandomBrightnessContrast(
                    brightness_limit=0.1, contrast_limit=0.1, p=0.3),
                A.Rotate(limit=2, border_mode=cv2.BORDER_CONSTANT,
                         value=255, p=0.2),
            ])

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        try: img = Image.open(self.crop_dir / p["crop_file"]).convert("RGB")
        except: img = Image.new("RGB", (64, 32), (255, 255, 255))

        if self.augment and hasattr(self, 'aug'):
            arr = np.array(img)
            arr = self.aug(image=arr)["image"]
            img = Image.fromarray(arr)

        pv = preprocess_crop_for_trocr(img)

        ids = encode(p["gt_text"])[:MAX_LEN]
        ids += [PAD_ID] * (MAX_LEN - len(ids))
        labels = torch.tensor(ids, dtype=torch.long)

        dec_inp = labels.clone()
        dec_inp = torch.roll(dec_inp, 1)
        dec_inp[0] = BOS_ID

        labels[labels == PAD_ID] = -100

        quality = torch.tensor(p.get("quality", 1.0), dtype=torch.float32)
        return pv, dec_inp, labels, quality


# ★ v5: quality-weighted loss (no label smoothing)
_loss_fn_none = nn.CrossEntropyLoss(ignore_index=-100, reduction='none')

def weighted_loss(logits, labels, weights):
    """Quality-weighted cross-entropy. weights shape: (B,)."""
    B, T, V = logits.shape
    per_tok = _loss_fn_none(logits.reshape(-1, V), labels.reshape(-1)).reshape(B, T)
    mask = (labels != -100).float()
    per_sample = (per_tok * mask).sum(1) / mask.sum(1).clamp(min=1)
    return (per_sample * weights).sum() / weights.sum()

def unweighted_loss(logits, labels):
    """Standard mean cross-entropy for validation."""
    B, T, V = logits.shape
    return nn.functional.cross_entropy(
        logits.reshape(-1, V), labels.reshape(-1), ignore_index=-100)


print("Dataset and loss functions ready.")

Dataset and loss functions ready.


## Phase A + Phase B Training

In [12]:
best_ckpt_path = CKPT_DIR / "trocr_ft_R1.pt"
overall_best_cer = baseline_cer
overall_best_info = {}

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PHASE A: Encoder Frozen, Exact-Match Pairs Only                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print("=" * 70)
print("PHASE A: Decoder warm-up (encoder frozen, exact-match pairs)")
print("=" * 70)

# Freeze encoder
for p in model.encoder.parameters():
    p.requires_grad = False

_enc_p = sum(p.numel() for p in model.encoder.parameters())
_dec_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Encoder params: {_enc_p:,} (frozen)")
print(f"  Trainable params: {_dec_p:,}")

# Filter exact-match pairs
phase_a_pairs = [p for p in pseudo_pairs if p["edit_ratio"] <= PHASE_A_MAX_EDIT]
_valid_a = [p for p in phase_a_pairs if (CROP_DIR_R1 / p["crop_file"]).exists()]
print(f"  Phase A pairs: {len(_valid_a):,} (edit ≤ {PHASE_A_MAX_EDIT})")

if len(_valid_a) >= 100:
    random.seed(42)
    _sh = list(_valid_a); random.shuffle(_sh)
    n_va = max(100, len(_sh) // 20)
    va_val_a, va_train_a = _sh[:n_va], _sh[n_va:]
    print(f"  Train: {len(va_train_a):,}  Val: {len(va_val_a):,}")

    opt_a = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=DECODER_LR_A, weight_decay=FT_WD)
    scaler_a = GradScaler("cuda")
    _total_steps_a = PHASE_A_EPOCHS * (len(va_train_a) // (FT_BS * GRAD_ACCUM) + 1)
    sched_a = get_scheduler("cosine", optimizer=opt_a,
                            num_warmup_steps=min(100, _total_steps_a // 10),
                            num_training_steps=_total_steps_a)

    val_ds_a = PseudoDS(va_val_a, CROP_DIR_R1, augment=False)
    val_dl_a = DataLoader(val_ds_a, batch_size=FT_BS, shuffle=False,
                          num_workers=2, pin_memory=True)

    best_cer_a = baseline_cer
    patience_a = 0

    for ep in range(1, PHASE_A_EPOCHS + 1):
        t0 = time.time()
        # Curriculum: epoch 1 sorted by quality desc, else random
        if ep == 1:
            td = sorted(va_train_a, key=lambda x: (-x.get("quality", 0), len(x["gt_text"])))
        else:
            td = random.sample(va_train_a, len(va_train_a))

        tr_ds = PseudoDS(td, CROP_DIR_R1, augment=True)
        tr_dl = DataLoader(tr_ds, batch_size=FT_BS, shuffle=False,
                           num_workers=2, pin_memory=True, drop_last=True)

        model.train()
        ep_loss, nb = 0.0, 0
        opt_a.zero_grad(set_to_none=True)

        for si, (pvs, dec_inp, labels, weights) in enumerate(tr_dl):
            pvs, dec_inp, labels = pvs.to(DEVICE), dec_inp.to(DEVICE), labels.to(DEVICE)
            with autocast("cuda"):
                out = model(pixel_values=pvs, decoder_input_ids=dec_inp)
                loss = unweighted_loss(out.logits, labels) / GRAD_ACCUM
            scaler_a.scale(loss).backward()
            if (si + 1) % GRAD_ACCUM == 0:
                scaler_a.unscale_(opt_a)
                nn.utils.clip_grad_norm_(model.parameters(), FT_GRAD_CLIP)
                scaler_a.step(opt_a); scaler_a.update(); sched_a.step()
                opt_a.zero_grad(set_to_none=True)
            ep_loss += loss.item() * GRAD_ACCUM; nb += 1

        avg_tl = ep_loss / max(nb, 1)

        # Validation loss
        model.eval()
        vl, nv = 0.0, 0
        with torch.inference_mode():
            for pvs, dec_inp, labels, _ in val_dl_a:
                pvs, dec_inp, labels = pvs.to(DEVICE), dec_inp.to(DEVICE), labels.to(DEVICE)
                with autocast("cuda"):
                    out = model(pixel_values=pvs, decoder_input_ids=dec_inp)
                    v = unweighted_loss(out.logits, labels)
                vl += v.item(); nv += 1
        avg_vl = vl / max(nv, 1)

        # Page CER
        q_cer, q_wer = quick_page_eval(n=20)

        mk = ""
        if q_cer < best_cer_a:
            best_cer_a = q_cer; patience_a = 0; mk = " ★CER"
            if q_cer < overall_best_cer:
                overall_best_cer = q_cer
                overall_best_info = {"phase": "A", "epoch": ep,
                                     "cer": q_cer, "wer": q_wer, "val_loss": avg_vl}
                torch.save({"model_state_dict": model.state_dict(),
                             **overall_best_info}, best_ckpt_path)
                mk += " →saved"
        else:
            patience_a += 1
            mk = f" (pat {patience_a}/{FT_PATIENCE})"

        print(f"  A ep{ep}/{PHASE_A_EPOCHS} tr={avg_tl:.4f} val={avg_vl:.4f} "
              f"CER={q_cer:.4f} WER={q_wer:.4f} "
              f"lr={opt_a.param_groups[0]['lr']:.2e} t={time.time()-t0:.0f}s{mk}")

        if patience_a >= FT_PATIENCE:
            print("  Early stop."); break

    print(f"\n  Phase A best CER: {best_cer_a:.4f}")
else:
    print("  Too few Phase A pairs, skipping.")
    best_cer_a = baseline_cer

# Reload best checkpoint before Phase B
if best_ckpt_path.exists():
    _b = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(_b["model_state_dict"])
    model.to(DEVICE)
    print(f"  Loaded best Phase A checkpoint (CER={_b.get('cer','?'):.4f})")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PHASE B: Full Fine-Tune, ALL Pairs, Quality-Weighted Loss             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("PHASE B: Full fine-tune (all params, all pairs, quality-weighted)")
print("=" * 70)

# Unfreeze encoder
for p in model.encoder.parameters():
    p.requires_grad = True
_total_t = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable params: {_total_t:,} (ALL)")

# ALL valid pairs
_valid_b = [p for p in pseudo_pairs if (CROP_DIR_R1 / p["crop_file"]).exists()]
print(f"  Phase B pairs: {len(_valid_b):,} (all, quality-weighted)")
if _valid_b:
    er_dist = [p["edit_ratio"] for p in _valid_b]
    print(f"  Edit ratio: mean={np.mean(er_dist):.3f}, "
          f"exact={sum(1 for e in er_dist if e==0):,}, "
          f">0.20={sum(1 for e in er_dist if e>0.20):,}")

if len(_valid_b) >= 100:
    random.seed(43)
    _sh = list(_valid_b); random.shuffle(_sh)
    n_vb = max(200, len(_sh) // 20)
    vb_val, vb_train = _sh[:n_vb], _sh[n_vb:]
    print(f"  Train: {len(vb_train):,}  Val: {len(vb_val):,}")

    # Differential LR: encoder gets lower LR
    enc_params = [p for n, p in model.named_parameters()
                  if n.startswith("encoder.") and p.requires_grad]
    dec_params = [p for n, p in model.named_parameters()
                  if not n.startswith("encoder.") and p.requires_grad]
    opt_b = optim.AdamW([
        {"params": enc_params, "lr": ENCODER_LR_B},
        {"params": dec_params, "lr": DECODER_LR_B},
    ], weight_decay=FT_WD)
    print(f"  Encoder LR: {ENCODER_LR_B}, Decoder LR: {DECODER_LR_B}")

    scaler_b = GradScaler("cuda")
    _total_steps_b = PHASE_B_EPOCHS * (len(vb_train) // (FT_BS * GRAD_ACCUM) + 1)
    sched_b = get_scheduler("cosine", optimizer=opt_b,
                            num_warmup_steps=min(300, _total_steps_b // 10),
                            num_training_steps=_total_steps_b)

    val_ds_b = PseudoDS(vb_val, CROP_DIR_R1, augment=False)
    val_dl_b = DataLoader(val_ds_b, batch_size=FT_BS, shuffle=False,
                          num_workers=2, pin_memory=True)

    best_cer_b = best_cer_a
    patience_b = 0

    for ep in range(1, PHASE_B_EPOCHS + 1):
        t0 = time.time()
        if ep == 1:
            td = sorted(vb_train, key=lambda x: (-x.get("quality", 0), len(x["gt_text"])))
        else:
            td = random.sample(vb_train, len(vb_train))

        tr_ds = PseudoDS(td, CROP_DIR_R1, augment=True)
        tr_dl = DataLoader(tr_ds, batch_size=FT_BS, shuffle=False,
                           num_workers=2, pin_memory=True, drop_last=True)

        model.train()
        ep_loss, nb = 0.0, 0
        opt_b.zero_grad(set_to_none=True)

        for si, (pvs, dec_inp, labels, weights) in enumerate(tr_dl):
            pvs = pvs.to(DEVICE)
            dec_inp = dec_inp.to(DEVICE)
            labels = labels.to(DEVICE)
            weights = weights.to(DEVICE)

            with autocast("cuda"):
                out = model(pixel_values=pvs, decoder_input_ids=dec_inp)
                # ★ v5: quality-weighted loss, no label smoothing
                loss = weighted_loss(out.logits, labels, weights) / GRAD_ACCUM

            scaler_b.scale(loss).backward()
            if (si + 1) % GRAD_ACCUM == 0:
                scaler_b.unscale_(opt_b)
                nn.utils.clip_grad_norm_(model.parameters(), FT_GRAD_CLIP)
                scaler_b.step(opt_b); scaler_b.update(); sched_b.step()
                opt_b.zero_grad(set_to_none=True)

            ep_loss += loss.item() * GRAD_ACCUM; nb += 1

        avg_tl = ep_loss / max(nb, 1)

        # Validation loss (unweighted)
        model.eval()
        vl, nv = 0.0, 0
        with torch.inference_mode():
            for pvs, dec_inp, labels, _ in val_dl_b:
                pvs, dec_inp, labels = pvs.to(DEVICE), dec_inp.to(DEVICE), labels.to(DEVICE)
                with autocast("cuda"):
                    out = model(pixel_values=pvs, decoder_input_ids=dec_inp)
                    v = unweighted_loss(out.logits, labels)
                vl += v.item(); nv += 1
        avg_vl = vl / max(nv, 1)

        # Page CER (★ v5: checkpoint by this, not loss)
        q_cer, q_wer = quick_page_eval(n=20)

        mk = ""
        if q_cer < best_cer_b:
            best_cer_b = q_cer; patience_b = 0; mk = " ★CER"
            if q_cer < overall_best_cer:
                overall_best_cer = q_cer
                overall_best_info = {"phase": "B", "epoch": ep,
                                     "cer": q_cer, "wer": q_wer, "val_loss": avg_vl}
                torch.save({"model_state_dict": model.state_dict(),
                             **overall_best_info}, best_ckpt_path)
                mk += " →saved"
        else:
            patience_b += 1
            mk = f" (pat {patience_b}/{FT_PATIENCE})"

        elr = opt_b.param_groups[0]['lr']
        dlr = opt_b.param_groups[1]['lr']
        print(f"  B ep{ep}/{PHASE_B_EPOCHS} tr={avg_tl:.4f} val={avg_vl:.4f} "
              f"CER={q_cer:.4f} WER={q_wer:.4f} "
              f"enc_lr={elr:.2e} dec_lr={dlr:.2e} t={time.time()-t0:.0f}s{mk}")

        if patience_b >= FT_PATIENCE:
            print("  Early stop."); break

    print(f"\n  Phase B best CER: {best_cer_b:.4f}")

# Reload overall best
if best_ckpt_path.exists():
    _b = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(_b["model_state_dict"])
    model.to(DEVICE).eval()
    print(f"\nOverall best: Phase {_b.get('phase','?')} ep{_b.get('epoch','?')} "
          f"CER={_b.get('cer','?'):.4f} WER={_b.get('wer','?'):.4f}")

PHASE A: Decoder warm-up (encoder frozen, exact-match pairs)
  Encoder params: 21,960,192 (frozen)
  Trainable params: 6,917,632
  Phase A pairs: 47,770 (edit ≤ 0.05)
  Train: 45,382  Val: 2,388
  A ep1/5 tr=0.1152 val=0.1095 CER=0.2958 WER=0.4397 lr=4.57e-05 t=492s ★CER →saved
  A ep2/5 tr=0.0911 val=0.0939 CER=0.2811 WER=0.4285 lr=3.34e-05 t=485s ★CER →saved
  A ep3/5 tr=0.0723 val=0.0910 CER=0.2764 WER=0.4222 lr=1.77e-05 t=485s ★CER →saved
  A ep4/5 tr=0.0613 val=0.0901 CER=0.2793 WER=0.4238 lr=4.93e-06 t=483s (pat 1/5)
  A ep5/5 tr=0.0554 val=0.0901 CER=0.2779 WER=0.4229 lr=6.30e-11 t=488s (pat 2/5)

  Phase A best CER: 0.2764
  Loaded best Phase A checkpoint (CER=0.2764)

PHASE B: Full fine-tune (all params, all pairs, quality-weighted)
  Trainable params: 28,877,824 (ALL)
  Phase B pairs: 63,199 (all, quality-weighted)
  Edit ratio: mean=0.091, exact=47,770, >0.20=12,395
  Train: 60,040  Val: 3,159
  Encoder LR: 2e-05, Decoder LR: 5e-05
  B ep1/12 tr=0.2616 val=0.5363 CER=0.2995 

## Save Artifacts + Quick Full Eval

In [13]:
print("=" * 70)
print("R1 SUMMARY")
print("=" * 70)

# Quick eval on 50 val pages
model.eval()
_cers, _wers = [], []
for _i in range(min(50, len(val_pages))):
    _gt = (val_pages[_i].get("text") or "").strip()
    if not _gt: continue
    try: _r = recognize_page(val_pages[_i])
    except: continue
    _m = compute_metrics(_r["text"], _gt)
    _cers.append(_m["cer"]); _wers.append(_m["wer"])
r1_cer = float(np.mean(_cers))
r1_wer = float(np.mean(_wers))
print(f"  R1 val (50 pages): CER={r1_cer:.4f}  WER={r1_wer:.4f}")
print(f"  Char Acc: {(1-r1_cer)*100:.2f}%  Word Acc: {(1-r1_wer)*100:.2f}%")
print(f"  Improvement: CER {baseline_cer:.4f} → {r1_cer:.4f} "
      f"({(1-r1_cer/baseline_cer)*100:+.1f}%)")

# Save artifacts for Part 2
artifacts = {
    "baseline_cer": baseline_cer,
    "baseline_wer": baseline_wer,
    "best_cer_r1": overall_best_cer,
    "best_wer_r1": overall_best_info.get("wer", r1_wer),
    "r1_pairs_count": len(pseudo_pairs),
    "char_to_token": char_to_token,
    "token_to_char": {str(k): v for k, v in token_to_char.items()},
    "actual_chars": actual_chars,
    "VOCAB_SIZE": VOCAB_SIZE,
    "PAD_ID": PAD_ID, "BOS_ID": BOS_ID, "EOS_ID": EOS_ID, "UNK_ID": UNK_ID,
}
with open(OUTPUT_DIR / "v5_part1_artifacts.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)
print(f"\nArtifacts saved to: {OUTPUT_DIR / 'v5_part1_artifacts.json'}")

print(f"\n{'='*70}")
print("FILES TO DOWNLOAD FOR PART 2:")
print(f"{'='*70}")
print(f"  1. {best_ckpt_path}")
print(f"     → Upload as Kaggle MODEL: 'v5-part1-model'")
print(f"  2. {OUTPUT_DIR / 'v5_part1_artifacts.json'}")
print(f"     → Upload as Kaggle DATASET: 'v5-part1-outputs'")
print(f"\nBoth files are in: {OUTPUT_DIR}")

R1 SUMMARY
  R1 val (50 pages): CER=0.2632  WER=0.4160
  Char Acc: 73.68%  Word Acc: 58.40%
  Improvement: CER 0.3166 → 0.2632 (+16.9%)

Artifacts saved to: /kaggle/working/phase4/v5_part1_artifacts.json

FILES TO DOWNLOAD FOR PART 2:
  1. /kaggle/working/phase4/checkpoints/trocr_ft_R1.pt
     → Upload as Kaggle MODEL: 'v5-part1-model'
  2. /kaggle/working/phase4/v5_part1_artifacts.json
     → Upload as Kaggle DATASET: 'v5-part1-outputs'

Both files are in: /kaggle/working/phase4
